# ukgeo — UK Geocoding Reference Data

This notebook demonstrates the [ukgeo](https://github.com/ThomasHSimm/ukgeo) Python geocoder
using the pre-built dataset hosted here on Kaggle.

ukgeo resolves messy UK location text — road references, junction names, place names,
colloquial names, old county names — to latitude/longitude coordinates. It is designed
for bulk offline processing without API calls or rate limits.

**Use cases:**
- Road safety analysis with STATS19 collision data
- Transport research requiring UK road network awareness
- Any application where UK location text needs geocoding at scale


In [1]:
import shutil, time
import pandas as pd
import polars as pl
from pathlib import Path
import subprocess

subprocess.run(
    ["pip", "install", "git+https://github.com/ThomasHSimm/ukgeo.git", "-q"],
    check=True
)

import ukgeo

# Copy Kaggle dataset to where ukgeo looks for it
data_dir = Path(ukgeo.__file__).parent.parent / "data"
data_dir.mkdir(parents=True, exist_ok=True)

src = Path("/kaggle/input/ukgeo-combined-dataset/ukgeo_data.parquet")
dataset_path = data_dir / "ukgeo_data.parquet"

if not dataset_path.exists():
    print(f"Copying dataset to {dataset_path}...")
    shutil.copy(src, dataset_path)
    print("Done")
else:
    print(f"Dataset already present")

from ukgeo import Geocoder
geo = Geocoder()
print("Geocoder ready")


Copying dataset to /usr/local/lib/python3.12/dist-packages/ukgeo/data/ukgeo_data.parquet...
Done
Geocoder ready


## Basic usage — single queries

ukgeo handles a wide range of UK location input types.


In [2]:
time_start = time.time()

test_cases = [
    ("Full postcode",           "LS1 1BA"),
    ("Motorway junction",       "M62 Junction 26"),
    ("A-road + place",          "A647 near Bradford"),
    ("Named interchange",       "Spaghetti Junction Birmingham"),
    ("Colloquial roundabout",   "Magic Roundabout Swindon"),
    ("Place + county",          "Skipton, North Yorkshire"),
    ("Typo",                    "Brafford West Yorkshire"),
    ("B-road + place",          "B6265 Bradford"),
    ("Old county name",         "Bournemouth Dorset"),
]

for label, text in test_cases:
    r = geo.geocode(text)
    status = f"{r.lat:.4f}, {r.lon:.4f}" if r.resolved else "unresolved"
    print(f"{label:<30} {text:<40} → {status} ({r.confidence})")

time_end = time.time()

print(f"Cell took {time_end - time_start:.2f}s or {(time_end - time_start) / len(test_cases):.2f}s per query")

Full postcode                  LS1 1BA                                  → 53.7971, -1.5564 (High)
Motorway junction              M62 Junction 26                          → 53.7362, -1.7266 (High)
A-road + place                 A647 near Bradford                       → 53.7925, -1.7240 (High)
Named interchange              Spaghetti Junction Birmingham            → 52.5135, -1.8487 (High)
Colloquial roundabout          Magic Roundabout Swindon                 → 51.5585, -1.7837 (High)
Place + county                 Skipton, North Yorkshire                 → 53.9602, -2.0177 (High)
Typo                           Brafford West Yorkshire                  → 53.7908, -1.7546 (Medium)
B-road + place                 B6265 Bradford                           → 53.8425, -1.8335 (High)
Old county name                Bournemouth Dorset                       → 50.7210, -1.8767 (High)
Cell took 1.44s or 0.16s per query


> **Note:** Named infrastructure aliases such as "Dartford Crossing" and
> "Lofthouse Interchange" now resolve via Level 0 (`data/infrastructure_aliases.csv`)
> before the OS Names scoring pipeline.


## Bulk geocoding

`geocode_batch()` processes a list or polars Series efficiently.
All OS data is loaded once at startup — subsequent queries are fast in-process lookups.

Dartford Crossing and Lofthouse Interchange now resolve at **Level 0** from the
curated alias CSV rather than Level 2 OS Names candidate scoring.


In [3]:
time_start = time.time()

locations = [
    "M1 Junction 42",
    "A64 York bypass near Tadcaster",
    "Dartford Crossing Kent",
    "Station Road Leeds",
    "Aberford West Yorkshire",
    "B1224 York",
    "Lofthouse Interchange",
    "WF10 4QH",
    "A1(M) Junction 47 Garforth",
    "Sighthill Edinburgh",
]

results = geo.geocode_batch(locations, show_progress=False)
print(results.select(["input", "lat", "lon", "confidence", "level_resolved", "interpreted_as"]))

time_end = time.time()
print(f"Cell took {time_end - time_start:.2f}s or {(time_end - time_start) / len(locations):.2f}s per query")

shape: (10, 6)
┌──────────────────────┬───────────┬───────────┬────────────┬────────────────┬─────────────────────┐
│ input                ┆ lat       ┆ lon       ┆ confidence ┆ level_resolved ┆ interpreted_as      │
│ ---                  ┆ ---       ┆ ---       ┆ ---        ┆ ---            ┆ ---                 │
│ str                  ┆ f64       ┆ f64       ┆ str        ┆ i64            ┆ str                 │
╞══════════════════════╪═══════════╪═══════════╪════════════╪════════════════╪═════════════════════╡
│ M1 Junction 42       ┆ 53.733815 ┆ -1.511212 ┆ High       ┆ 2              ┆ M1 J42 (Junction)   │
│ A64 York bypass near ┆ 53.884747 ┆ -1.26405  ┆ High       ┆ 2              ┆ Tadcaster (Town)    │
│ Tadcaster            ┆           ┆           ┆            ┆                ┆                     │
│ Dartford Crossing    ┆ 51.456099 ┆ 0.246748  ┆ High       ┆ 2              ┆ Dartford (Town)     │
│ Kent                 ┆           ┆           ┆            ┆               

## Performance benchmark

Benchmarked against 5,000 STATS19 2024 collision records — real road reference inputs
with verified ground truth coordinates.


In [4]:
benchmark_data = {
    "Metric": ["Resolve rate", "Median error", "Within 500m", "Within 5km", "B-roads resolved"],
    "v0.2": ["68.9%", "4,593m", "9.7%", "51.6%", "5.1%"],
    "v0.3": ["99.9%", "3,299m", "~10%", "59.9%", "99.9%"],
    "v0.4": ["99.9%", "3,299m", "~10%", "59.9%", "99.9%"],
}
df = pd.DataFrame(benchmark_data)
print(df.to_string(index=False))
print("
Note: median error reflects road centroid resolution.")
print("STATS19 records already have GPS coordinates — ukgeo is most useful")
print("for derived datasets that have road references but no coordinates.")


          Metric v0.1   v0.2   v0.3
    Resolve rate    —  68.9%  99.9%
    Median error    — 4,593m 3,299m
     Within 500m    —   9.7%   ~10%
      Within 5km    —  51.6%  59.9%
B-roads resolved    —   5.1%  99.9%

Note: median error reflects road centroid resolution.
STATS19 records already have GPS coordinates — ukgeo is most useful
for derived datasets that have road references but no coordinates.


## What's in the dataset


In [5]:
data = pl.read_parquet(dataset_path)

print(f"Total rows: {len(data):,}")
print(f"Built:      {data['BUILT_AT'][0]}")
print()
print("By source:")
print(data.group_by("SOURCE").agg(pl.len().alias("rows")).sort("rows", descending=True))
print()
print("By LOCAL_TYPE (top 15):")
print(data.group_by("LOCAL_TYPE").agg(pl.len().alias("rows")).sort("rows", descending=True).head(15))


Total rows: 2,809,377
Built:      2026-05-17

By source:
shape: (4, 2)
┌───────────────┬─────────┐
│ SOURCE        ┆ rows    │
│ ---           ┆ ---     │
│ str           ┆ u32     │
╞═══════════════╪═════════╡
│ os_open_names ┆ 2670589 │
│ osm_roads     ┆ 105720  │
│ osm           ┆ 32399   │
│ os_open_roads ┆ 669     │
└───────────────┴─────────┘

By LOCAL_TYPE (top 15):
shape: (13, 2)
┌───────────────────┬─────────┐
│ LOCAL_TYPE        ┆ rows    │
│ ---               ┆ ---     │
│ str               ┆ u32     │
╞═══════════════════╪═════════╡
│ Postcode          ┆ 1741938 │
│ Named Road        ┆ 880223  │
│ B Road            ┆ 105720  │
│ Named Roundabout  ┆ 28659   │
│ Village           ┆ 15155   │
│ …                 ┆ …       │
│ Named Junction    ┆ 3740    │
│ Other Settlement  ┆ 2913    │
│ Town              ┆ 1353    │
│ Motorway Junction ┆ 669     │
│ City              ┆ 71      │
└───────────────────┴─────────┘


## Further reading

- [ukgeo GitHub repo](https://github.com/ThomasHSimm/ukgeo) — source code, documentation, download scripts
- [Open Road Risk](https://github.com/ThomasHSimm/open-road-risk) — road safety risk modelling pipeline
- [Open Road Risk Website](https://openroadrisk.org/) — road safety risk modelling website
- [docs/alternative.md](https://github.com/ThomasHSimm/ukgeo/blob/main/docs/alternative.md) — comparison with other UK geocoding tools
- [docs/STATUS.md](https://github.com/ThomasHSimm/ukgeo/blob/main/docs/STATUS.md) — current test results

**Data sources:**
- OS Open Names © Crown Copyright and database right 2024 (OGL v3)
- OS Open Roads © Crown Copyright and database right 2024 (OGL v3)
- © OpenStreetMap contributors (ODbL v1.0)
